<a href="https://colab.research.google.com/github/ngophuctaivp20-crypto/Final-Project-/blob/main/Saving_Goal_Coach_Final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

6 Steps Process:

1.understand the problem

Many people they want to save their money to buy things such as laptop, watch, or something valuable.But they don't know how to create a good saving plan. Thus, I will introduce the Saving Goal Coach that let the user set goals with deadlines, track progress,
analyse spending from a CSV of transactions, and ask an AI coach for advice based on their real numbers. It gives general budgeting guidance, not financial
advice.

2.identify the inputs and outputs

The inputs should be a savings goal, which consists of a goal name, a target amount, a deadline in YYYY-MM-DD format and any savings the user already holds. Therefore, I want the system produces output like a summary showing the amount saved, the percentage complete, the number of months remaining, the monthly contribution required to meet the deadline and an overall progress status.

Other should be transaction data, supplied as a CSV file containing the date, description, category and amount of each transaction, with income recorded as a positive value and spending as a negative value. Output would be spending totals by category, a monthly cash flow summary and a set of recommended spending reductions, together with an overall assessment of whether the user's goals are achievable.

Additionally, another input is that the user can communicate with AI and the output it will provide back is their suggesstion whether the user's budget is on tract or not and help the user what to do to achieve the goal.

3.work an example by hand,

Goal: Laptop, target 2,000, saved 500, created 26 Sep 2026, deadline 26 Mar 2027 (181 days).
- Remaining = 1500
- Months left = 181 ÷ 30 = 6 (rounded to 1 decimal)
- Needed per month = _250_
- Progress % = 25%
- Status = On track
Ask Gemini AI for the plan: Example like, "Hi, what should I do to get money in order to achieve my goal?"

4.write pseudocode

1. Settings

SET APP_CURRENCY to "AUD"
SET REQUIRED_COLUMNS to [date, description, category, amount]
SET DAYS_PER_MONTH to 30
SET CATEGORY_RULES to a dictionary: category → {cut percentage, tip}
SET DEFAULT_RULE to {cut 10%, "Review what's going into this category."}

FUNCTION load_api_key()
    TRY
        RETURN the key "GEMINI_API_KEY" from Colab Secrets
    IF that fails
        RETURN the key from the environment variables (or nothing)

SET GEMINI_API_KEY to load_api_key()
IF GEMINI_API_KEY exists
    PRINT "Gemini connected"
ELSE
    PRINT "Chat will use the offline coach"


2. Helper functions

FUNCTION savings_goal_calculator(target, saved, months)
    TRY
        turn target, saved and months into numbers
    IF that fails
        RETURN {error: "must all be numbers"}
    IF target <= 0 → RETURN {error: "Target must be more than 0"}
    IF months <= 0 → RETURN {error: "Months must be more than 0"}

    remaining ← larger of (target − saved) and 0
    per_month ← remaining ÷ months
    per_week ← per_month × 12 ÷ 52
    percent_done ← smaller of (saved ÷ target × 100) and 100
    RETURN {remaining, per_month, per_week, percent_done}

FUNCTION plan_text(plan)
    IF "error" is in plan → RETURN the error message
    IF remaining = 0      → RETURN "You have already reached your goal!"
    RETURN "You still need $X (Y% done). Save $A a month, about $B a week."

SET OPERATIONS to {"1": "How much should I save each month?", "2": "How many months will it take?"}

FUNCTION perform_calculation(operation, target, saved, value)
    IF value <= 0 → RETURN "The last number must be more than 0."
    IF operation = "1"
        RETURN plan_text(savings_goal_calculator(target, saved, value))
    ELSE IF operation = "2"
        months ← 0
        WHILE saved < target
            saved ← saved + value
            months ← months + 1
        RETURN "It will take " + months + " months."
    ELSE
        RETURN "Unknown question."

FUNCTION get_number(prompt)
    REPEAT forever
        TRY
            RETURN what the user types, turned into a number
        IF it isn't a number
            PRINT "Please type a number."

FUNCTION get_operation()
    REPEAT forever
        FOR EACH key and label in OPERATIONS
            PRINT key and label
        choice ← what the user types
        IF choice = "q" OR choice is in OPERATIONS
            RETURN choice
        PRINT "That isn't an option."

FUNCTION run_calculator()
    REPEAT forever
        operation ← get_operation()
        IF operation = "q"
            PRINT "Bye!"
            STOP the loop (break)
        target ← get_number("Target amount")
        saved ← get_number("Already saved")
        IF operation = "1"
            value ← get_number("Number of months")
        ELSE
            value ← get_number("Amount saved each month")
        PRINT perform_calculation(operation, target, saved, valu

3. Section 3: Goals (nested dictionaries)

goals ← empty dictionary          (goal name in lower case → goal dictionary)

FUNCTION total_saved(goal)
    total ← 0
    FOR EACH record in goal's contributions
        total ← total + record's amount
    RETURN total

FUNCTION percent_done(goal)
    RETURN smaller of (total_saved ÷ target × 100) and 100

FUNCTION monthly_needed(goal)
    RETURN larger of (target − total_saved) and 0, divided by months

FUNCTION status(goal)
    IF percent >= 100      → RETURN "Complete"
    ELSE IF percent >= 50  → RETURN "More than halfway"
    ELSE IF percent > 0    → RETURN "Started"
    ELSE                   → RETURN "Not started"

FUNCTION add_goal(name, target, months, saved)
    TRY
        turn target, months and saved into numbers
    IF that fails → RETURN "must all be numbers"
    IF name is empty               → RETURN "Please give your goal a name."
    IF target <= 0                 → RETURN "Target must be more than 0."
    IF months <= 0                 → RETURN "Months must be more than 0."
    IF saved < 0                   → RETURN "Already saved can't be negative."
    IF name (lower case) is in goals → RETURN "You already have that goal."
    goal ← {name, target, months, contributions: empty list}
    IF saved > 0
        ADD {saved, "Starting balance"} to the goal's contributions
    STORE goal in goals
    RETURN "Added goal. Save $X each month."

FUNCTION add_money(name, amount)
    IF name (lower case) is not in goals → RETURN "Please pick a goal."
    TRY
        turn amount into a number
    IF that fails → RETURN "Amount must be a number."
    IF amount = 0 OR total_saved + amount < 0
        RETURN "Amount can't be 0, or more than you have saved."
    ADD {amount} to the goal's contributions
    RETURN "Goal is now X% done."

FUNCTION goal_names()
    FOR EACH goal in goals → collect its name
    RETURN the list of names

FUNCTION total_monthly_needed()
    total ← 0
    FOR EACH goal in goals
        total ← total + monthly_needed(goal)
    RETURN total

FUNCTION goals_text()
    IF there are no goals → RETURN "You don't have any goals yet."
    FOR EACH goal → add a line: name, % done, status, amount per month
    RETURN the text

FUNCTION goals_table()
    FOR EACH goal → make a row
    RETURN the rows as a pandas table

4. Section 4: pandas, reading the CSV

FUNCTION load_transactions(path)
    TRY
        table ← read the CSV file
    IF that fails → RETURN False
    MAKE the column names lower case
    FOR EACH column in [date, category, amount]
        IF column is not in the table → RETURN False
    TURN the amount column into numbers (bad ones become empty)
    REMOVE rows with an empty amount
    ADD a month column, e.g. "2026-07"
    app_data["table"] ← table
    RETURN True

FUNCTION total_for(table, category)
    total ← 0
    FOR i IN range(number of rows)
        row ← row number i
        IF row's category = category AND row's amount < 0
            total ← total − amount
    RETURN total

FUNCTION spending_by_category(table)
    totals ← empty dictionary
    FOR EACH different category in the table
        spent ← total_for(table, category)
        IF spent > 0 → totals[category] ← spent
    RETURN totals

FUNCTION monthly_surplus(table)
    RETURN sum of all amounts ÷ number of months

FUNCTION suggest_cuts(table)
    surplus ← monthly_surplus(table)
    needed ← total_monthly_needed()
    text ← "Money left over each month" and "Your goals need"
    IF surplus >= needed → add "you can afford your goals"
    ELSE                 → add "you are $X short each month"
    FOR EACH category and amount spent
        IF category is not in CATEGORY_TIPS → SKIP (continue)
        saving ← spent ÷ months × the cut percentage
        IF saving < 5 → SKIP (continue)
        ADD a line with the category, the saving and the tip
    RETURN text

5. Section 5: requests, currency and the Gemini chat with custom tools

FUNCTION convert_currency(amount, from_code, to_code)
    TRY
        turn amount into a number
    IF that fails → RETURN "Amount must be a number."
    MAKE both codes upper case
    IF from_code = to_code → RETURN the same amount
    TRY
        ASK the Frankfurter website for the exchange rate
    IF that fails → RETURN "Couldn't convert."
    RETURN "amount FROM = converted TO"

SET GEMINI_TOOLS to a list of 5 tool descriptions:
    get_goals, add_money, calculate_plan, spending_tips, convert_currency

FUNCTION run_tool(name, inputs)
    IF name = "get_goals"             → RETURN goals_text()
    ELSE IF name = "add_money"        → RETURN add_money(goal name, amount)
    ELSE IF name = "calculate_plan"   → RETURN savings_goal_calculator(target, saved, months)
    ELSE IF name = "spending_tips"
        IF "table" is not in app_data → RETURN "No spending data yet."
        RETURN suggest_cuts(table)
    ELSE IF name = "convert_currency" → RETURN convert_currency(amount, from, to)
    ELSE                              → RETURN "Unknown tool"

FUNCTION ask_gemini(message)
    contents ← [the user's message]
    FOR round IN range(3)                          (at most 3 rounds)
        body ← {instructions, GEMINI_TOOLS, contents}

        FOR attempt IN range(3)                    (try again if Gemini is busy)
            TRY
                SEND body to Gemini
            IF that fails → RETURN "Gemini error: couldn't connect"
            IF the status code is not 503 → STOP trying (break)
            WAIT 3 seconds

        IF the status code is not 200 → RETURN "Gemini error" + code + reason
        TRY
            parts ← the parts of Gemini's reply
        IF that fails → RETURN "Gemini error: empty reply"
        ADD Gemini's reply to contents

        tool_answers ← empty list
        FOR EACH part in parts
            IF Gemini asked for a tool
                result ← run_tool(tool name, tool inputs)
                PRINT "Tool used: name → result"
                ADD the result to tool_answers

        IF tool_answers is empty                   (no tools needed)
            answer ← all the text parts joined together
            RETURN answer

        ADD tool_answers to contents               (send results back, go round again)
    RETURN "Sorry, that question was too complicated."

FUNCTION offline_reply(message)
    text ← message in lower case
    IF text contains "goal" OR "progress"          → RETURN run_tool("get_goals")
    ELSE IF text contains "spend" OR "save" OR "tip" → RETURN run_tool("spending_tips")
    ELSE → RETURN "Offline mode: ask about your goals or tips to save."

FUNCTION coach_reply(message, history)
    IF GEMINI_API_KEY is not ""
        answer ← ask_gemini(message)
        IF answer does not start with "Gemini error"
            RETURN answer
        PRINT the error in Colab
    RETURN offline_reply(message)                  (no key, or Gemini failed)

6. Section 6: Sample data

FUNCTION make_sample_csv()
    one_month ← list of 7 transactions (description, category, amount)
    rows ← empty list
    FOR EACH month in ["2026-07", "2026-08", "2026-09"]
        FOR EACH transaction in one_month
            ADD {date, description, category, amount} to rows
    SAVE rows as "sample_transactions.csv"
    RETURN the file name

7. Section 7: Tests

FUNCTION run_tests()
    CHECK money() formats numbers correctly
    CHECK savings_goal_calculator gives the right plan, and errors for bad input
    CHECK perform_calculation answers both questions
    CHECK add_goal and add_money work, and reject bad input
    WRITE a small test CSV file
    CHECK load_transactions removes bad rows, and returns False for a missing file
    CHECK total_for, spending_by_category, monthly_surplus and suggest_cuts
    CHECK convert_currency (the parts that don't need the internet)
    CHECK every custom tool in run_tool, including an unknown tool name
    CLEAR goals and app_data
    PRINT "All tests passed"

8. Section 8: Gradio app

FUNCTION ui_add_goal(name, target, months, saved)
    RETURN add_goal message, goals table, updated goal dropdown

FUNCTION ui_add_money(name, amount)
    RETURN add_money message, goals table

FUNCTION ui_load(path)
    IF load_transactions(path) = False → RETURN "Couldn't read that file"
    RETURN "Loaded X transactions", spending table, suggest_cuts text

FUNCTION ui_sample()
    RETURN ui_load(make_sample_csv())

FUNCTION ui_calculator(target, saved, months)
    RETURN plan_text(savings_goal_calculator(target, saved, months))

BUILD the window with 5 tabs:
    Goals       → add a goal, add money, goals table
    Calculator  → goal amount, saved, months, Calculate button
    Spending    → upload CSV, sample data button, spending table, suggestions
    Currency    → amount, from, to, Convert button
    Coach       → chat box that uses coach_reply
CONNECT each button to its function
LAUNCH the app

  

5.convert pseudocode to Python



6.test with a variety of data.

5.convert pseudocode to Python


In [ ]:
!pip -q install -U gradio pandas requests

## 1. Imports and settings

In [ ]:
import pandas as pd
import requests
import gradio as gr

CURRENCY_URL = "https://api.frankfurter.app/latest"
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent"

# Nested dictionary: categories we could cut, how much (0.30 = 30%) and a tip
CATEGORY_TIPS = {
    "Dining Out":    {"cut": 0.30, "tip": "Cook at home more often."},
    "Entertainment": {"cut": 0.25, "tip": "Look for free events and student discounts."},
    "Shopping":      {"cut": 0.30, "tip": "Wait 48 hours before buying something."},
    "Subscriptions": {"cut": 0.50, "tip": "Cancel the ones you don't use."},
    "Groceries":     {"cut": 0.10, "tip": "Plan meals and shop with a list."},
}

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = ""         # no key: the chat will use the offline coach

app_data = {}                   # app_data["table"] will hold the transactions once a CSV is loaded

##2: Custom Tools: Saving Goal Calculator


In [ ]:
def money(amount):
    return f"${amount:.2f}"      # .2f means 2 decimal places, e.g. $12.50


def savings_goal_calculator(target, saved, months):
    """MY CUSTOM TOOL: takes a goal, the money already saved and the months left,
    and gives back a savings plan as a dictionary."""
    try:
        target = float(target)
        saved = float(saved)
        months = float(months)
    except Exception:
        return {"error": "Target, saved and months must all be numbers."}

    if target <= 0:
        return {"error": "Target must be more than 0."}
    if months <= 0:
        return {"error": "Months must be more than 0."}

    remaining = max(target - saved, 0)              # money still needed (never below 0)
    per_month = remaining / months
    per_week = per_month * 12 / 52                  # 12 months = 52 weeks
    percent_done = min(saved / target * 100, 100)   # never above 100%

    return {
        "remaining": round(remaining, 2),
        "per_month": round(per_month, 2),
        "per_week": round(per_week, 2),
        "percent_done": round(percent_done, 1),
    }


def plan_text(plan):
    """Turn the calculator's dictionary into sentences for the user."""
    if "error" in plan:
        return plan["error"]
    if plan["remaining"] == 0:
        return "You have already reached your goal! 🎉"
    text = "You still need " + money(plan["remaining"]) + " (" + str(plan["percent_done"]) + "% done).\n"
    text = text + "Save " + money(plan["per_month"]) + " a month, which is about " + money(plan["per_week"]) + " a week."
    return text


OPERATIONS = {
    "1": "How much should I save each month?",
    "2": "How many months will it take?",
}


def perform_calculation(operation, target, saved, value):
    """
    operation 1: value = number of months       -> how much to save each month
    operation 2: value = amount saved per month -> how many months it takes
    Gives back the answer as a sentence.
    """
    if value <= 0:
        return "The last number must be more than 0."
    if operation == "1":
        return plan_text(savings_goal_calculator(target, saved, value))     # uses my custom tool
    elif operation == "2":
        months = 0
        while saved < target:
            saved = saved + value
            months = months + 1
        return "It will take " + str(months) + " months."
    else:
        return "Unknown question."


def get_number(prompt):
    """Keep asking until the user types a number."""
    while True:
        try:
            return float(input(prompt))
        except ValueError:
            print("Please type a number.")


def get_operation():
    """Keep asking until the user types 1, 2 or q."""
    while True:
        for key, label in OPERATIONS.items():
            print(key, "-", label)
        choice = input("Choose 1 or 2 (q to quit): ").strip().lower()
        if choice == "q" or choice in OPERATIONS:
            return choice
        print("That isn't an option.")


def run_calculator():
    while True:
        operation = get_operation()
        if operation == "q":
            print("Bye!")
            break
        target = get_number("Target amount: ")
        saved = get_number("Already saved: ")
        if operation == "1":
            value = get_number("Number of months: ")
        else:
            value = get_number("Amount saved each month: ")
        print(perform_calculation(operation, target, saved, value))


# run_calculator()      # remove the # to try the calculator

#3: Goals (nested dictionaries)


In [ ]:
goals = {}      # goal name in lower case -> the goal's dictionary


def total_saved(goal):
    total = 0
    for record in goal["contributions"]:
        total = total + record["amount"]
    return total


def percent_done(goal):
    return round(min(total_saved(goal) / goal["target"] * 100, 100), 1)


def monthly_needed(goal):
    return round(max(goal["target"] - total_saved(goal), 0) / goal["months"], 2)


def status(goal):
    percent = percent_done(goal)
    if percent >= 100:
        return "Complete"
    elif percent >= 50:
        return "More than halfway"
    elif percent > 0:
        return "Started"
    else:
        return "Not started"


def add_goal(name, target, months, saved=0):
    """Checks the input, adds the goal, and gives back a message for the user."""
    name = str(name).strip()
    try:
        target = float(target)
        months = float(months)
        saved = float(saved)
    except Exception:                    # e.g. "abc" or an empty box
        return "Target, months and already saved must all be numbers."

    if name == "":
        return "Please give your goal a name."
    if target <= 0:
        return "Target must be more than 0."
    if months <= 0:
        return "Months must be more than 0."
    if saved < 0:
        return "Already saved can't be negative."
    if name.lower() in goals:
        return "You already have a goal called " + name + "."

    goal = {"name": name, "target": target, "months": months, "contributions": []}
    if saved > 0:
        goal["contributions"].append({"amount": saved, "note": "Starting balance"})
    goals[name.lower()] = goal
    return "Added " + name + ". Save " + money(monthly_needed(goal)) + " each month."


def add_money(name, amount):
    key = str(name).strip().lower()
    if key not in goals:
        return "Please pick a goal."
    try:
        amount = float(amount)
    except Exception:
        return "Amount must be a number."

    goal = goals[key]
    if amount == 0 or total_saved(goal) + amount < 0:
        return "Amount can't be 0, or more than you have saved."
    goal["contributions"].append({"amount": amount, "note": ""})
    return goal["name"] + " is now " + str(percent_done(goal)) + "% done."


def goal_names():
    names = []
    for goal in goals.values():
        names.append(goal["name"])
    return names


def total_monthly_needed():
    total = 0
    for goal in goals.values():
        total = total + monthly_needed(goal)
    return round(total, 2)


def goals_text():
    if len(goals) == 0:
        return "You don't have any goals yet."
    text = ""
    for goal in goals.values():
        text = text + goal["name"] + ": " + str(percent_done(goal)) + "% done, " + status(goal)
        text = text + ", save " + money(monthly_needed(goal)) + " a month\n"
    return text


def goals_table():
    rows = []
    for goal in goals.values():
        rows.append({"Goal": goal["name"], "Target": goal["target"], "Saved": total_saved(goal),
                     "Percent": percent_done(goal), "Per month": monthly_needed(goal), "Status": status(goal)})
    return pd.DataFrame(rows)

#4: Pandas: reading the CSV


In [ ]:
def load_transactions(path):
    """Reads the CSV into app_data["table"]. Gives back True if it worked, False if not."""
    try:
        table = pd.read_csv(path)
    except Exception:
        return False

    table.columns = table.columns.str.lower()
    for column in ["date", "category", "amount"]:
        if column not in table.columns:
            return False

    table["amount"] = pd.to_numeric(table["amount"], errors="coerce")   # bad amounts become empty
    table = table.dropna(subset=["amount"])                             # remove those rows
    table["month"] = table["date"].astype(str).str[:7]                  # "2026-07-15" -> "2026-07"
    app_data["table"] = table
    return True


def total_for(table, category):
    """Total spent in one category."""
    total = 0
    for i in range(len(table)):
        row = table.iloc[i]
        if row["category"] == category and row["amount"] < 0:
            total = total - row["amount"]
    return round(total, 2)


def spending_by_category(table):
    totals = {}
    for category in table["category"].unique():
        spent = total_for(table, category)
        if spent > 0:
            totals[category] = spent
    return totals


def monthly_surplus(table):
    """Average money left over each month (income minus spending)."""
    months = len(table["month"].unique())
    return round(table["amount"].sum() / months, 2)


def suggest_cuts(table):
    surplus = monthly_surplus(table)
    needed = total_monthly_needed()
    months = len(table["month"].unique())

    text = "Money left over each month: " + money(surplus) + "\n"
    text = text + "Your goals need: " + money(needed) + " a month\n"
    if surplus >= needed:
        text = text + "Good news: you can afford your goals!\n"
    else:
        text = text + "You are " + money(needed - surplus) + " short each month.\n"

    text = text + "\nIdeas to save more:\n"
    for category, spent in spending_by_category(table).items():
        if category not in CATEGORY_TIPS:
            continue                     # e.g. rent: we can't cut it
        rule = CATEGORY_TIPS[category]
        saving = round(spent / months * rule["cut"], 2)
        if saving < 5:
            continue                     # too small to matter
        text = text + "- " + category + ": save about " + money(saving) + " a month. " + rule["tip"] + "\n"
    return text

#5: requests: currency converter and Gemini chat with custom tools



In [ ]:
def convert_currency(amount, from_code, to_code):
    """Gives back the answer as a sentence."""
    try:
        amount = float(amount)
    except Exception:
        return "Amount must be a number."
    from_code = str(from_code).strip().upper()
    to_code = str(to_code).strip().upper()
    if from_code == to_code:
        return str(amount) + " " + from_code + " = " + str(amount) + " " + to_code

    try:
        response = requests.get(CURRENCY_URL, params={"from": from_code, "to": to_code}, timeout=10)
        rate = response.json()["rates"][to_code]
    except Exception:                   # no internet, or a currency code that doesn't exist
        return "Couldn't convert. Check the currency codes and your internet."
    return str(amount) + " " + from_code + " = " + str(round(amount * rate, 2)) + " " + to_code


# Each tool: a name, a description, and (if needed) the inputs it takes
GEMINI_TOOLS = [
    {"name": "get_goals",
     "description": "Get the user's savings goals: percent done, status and amount needed each month."},
    {"name": "add_money",
     "description": "Add money to one of the user's goals (negative to take money out).",
     "parameters": {"type": "object",
                    "properties": {"goal_name": {"type": "string"}, "amount": {"type": "number"}},
                    "required": ["goal_name", "amount"]}},
    {"name": "calculate_plan",
     "description": "Savings-goal calculator: work out how much is left and how much to save each month and week to reach a target.",
     "parameters": {"type": "object",
                    "properties": {"target": {"type": "number"}, "saved": {"type": "number"},
                                   "months": {"type": "number"}},
                    "required": ["target", "months"]}},
    {"name": "spending_tips",
     "description": "Get the user's money left over each month and ideas to cut spending."},
    {"name": "convert_currency",
     "description": "Convert money from one currency to another, e.g. USD to AUD.",
     "parameters": {"type": "object",
                    "properties": {"amount": {"type": "number"}, "from_currency": {"type": "string"},
                                   "to_currency": {"type": "string"}},
                    "required": ["amount", "from_currency", "to_currency"]}},
]


def run_tool(name, inputs):
    """Run the tool with this name. inputs is a dictionary, e.g. {"goal_name": "Trip", "amount": 50}"""
    if name == "get_goals":
        return goals_text()

    elif name == "add_money":
        return add_money(inputs.get("goal_name", ""), inputs.get("amount", ""))

    elif name == "calculate_plan":
        return savings_goal_calculator(inputs.get("target", ""), inputs.get("saved", 0), inputs.get("months", ""))

    elif name == "spending_tips":
        if "table" not in app_data:
            return "No spending data yet. Load a CSV in the Spending tab first."
        return suggest_cuts(app_data["table"])

    elif name == "convert_currency":
        return convert_currency(inputs.get("amount", ""), inputs.get("from_currency", ""), inputs.get("to_currency", ""))

    else:
        return "Unknown tool: " + name


COACH_INSTRUCTIONS = ("You are a friendly savings coach for a university student. Amounts are in AUD. "
                      "Always use your tools to get real numbers - never make numbers up. Keep answers short.")


def ask_gemini(message):
    contents = [{"role": "user", "parts": [{"text": message}]}]     # the conversation so far

    for i in range(3):        # at most 3 rounds, so it can never loop forever
        body = {"system_instruction": {"parts": [{"text": COACH_INSTRUCTIONS}]},
                "tools": [{"function_declarations": GEMINI_TOOLS}],
                "contents": contents}
        try:
            response = requests.post(GEMINI_URL, headers={"x-goog-api-key": GEMINI_API_KEY}, json=body, timeout=30)
            parts = response.json()["candidates"][0]["content"]["parts"]
        except Exception:
            return "Sorry, I couldn't reach Gemini. Check your API key and internet."
        contents.append({"role": "model", "parts": parts})

        # Step 3: run every tool Gemini asked for
        tool_answers = []
        for part in parts:
            if "functionCall" in part:
                tool_name = part["functionCall"]["name"]
                tool_inputs = part["functionCall"].get("args", {})    # "args" is Gemini's word for the inputs
                result = run_tool(tool_name, tool_inputs)
                print("Tool used:", tool_name, "->", result)          # shows in Colab, handy for checking
                tool_answers.append({"functionResponse": {"name": tool_name, "response": {"result": result}}})

        # No tools asked for: this is Gemini's final answer
        if len(tool_answers) == 0:
            answer = ""
            for part in parts:
                answer = answer + part.get("text", "")
            return answer

        # Step 4: send the tool results back to Gemini and go round again
        contents.append({"role": "user", "parts": tool_answers})

    return "Sorry, that question was too complicated. Try a simpler one."


def offline_reply(message):
    """Without a Gemini key: look for keywords and use the same tools."""
    text = message.lower()
    if "goal" in text or "progress" in text:
        return run_tool("get_goals", {})
    elif "spend" in text or "save" in text or "tip" in text:
        return run_tool("spending_tips", {})
    else:
        return "Offline mode: ask me about your goals or tips to save."


def coach_reply(message, history):
    if GEMINI_API_KEY != "":
        return ask_gemini(message)
    return offline_reply(message)


#6. Sample Data


In [ ]:
def make_sample_csv():
    one_month = [
        ("Part-time job", "Income", 2300),
        ("Rent", "Rent", -720),
        ("Supermarket", "Groceries", -370),
        ("Takeaway", "Dining Out", -155),
        ("Streaming and gym", "Subscriptions", -67),
        ("Cinema", "Entertainment", -87),
        ("Clothes", "Shopping", -115),
    ]
    rows = []
    for month in ["2026-07", "2026-08", "2026-09"]:
        for description, category, amount in one_month:
            rows.append({"date": month + "-01", "description": description,
                         "category": category, "amount": amount})
    pd.DataFrame(rows).to_csv("sample_transactions.csv", index=False)
    return "sample_transactions.csv"

#7:Test with Assert

